# EDI Analytics Engineer Exercise — Data Profile and Quality Review

This notebook documents the initial investigation of the four synthetic CSV files supplied with the EDI analytics engineer exercise. The goal is not to clean everything immediately; the goal is to understand the data, identify governance decisions, and define what is safe to use in a progression indicator.

Audience note: director-facing outputs should use learner IDs rather than learner names and should present evidence neutrally so the program director can decide what follow-up is appropriate.


## Investigation approach

- Load the raw CSVs from `data/raw/`.
- Apply candidate normalization rules without overwriting the source values.
- Count duplicate keys, missing values, orphan references, date format issues, competency-domain aliases, and score classes.
- Assess whether a cohort-relative competency progression indicator is feasible.


In [1]:
import csv
import re
from collections import Counter, defaultdict
from datetime import datetime
from pathlib import Path

RAW = Path("../data/raw") if Path("../data/raw").exists() else Path("data/raw")


def read_csv(name):
    with (RAW / name).open(newline="", encoding="utf-8-sig") as handle:
        return list(csv.DictReader(handle))

learners = read_csv("learners.csv")
sessions = read_csv("curricular_experiences.csv")
crosswalk = read_csv("competency_crosswalk.csv")
events = read_csv("assessment_events.csv")


def normalize_learner_id(value):
    s = (value or "").strip().upper().replace("-", "")
    if re.fullmatch(r"\d+", s):
        return f"VU{int(s):04d}"
    match = re.fullmatch(r"VU(\d+)", s)
    if match:
        return f"VU{int(match.group(1)):04d}"
    return s


def normalize_session_id(value):
    return (value or "").strip().upper()


def normalize_cohort(value):
    s = (value or "").strip().upper()
    match = re.search(r"(20\d{2})", s)
    return match.group(1) if match else ""


DOMAIN_ALIASES = {
    "KP": "Knowledge for Practice",
    "PC": "Patient Care",
    "IPC": "Interprofessional Collaboration",
    "PROFESSIONALISM": "Professionalism",
    "PATIENT CARE": "Patient Care",
    "INTERPERSONAL AND COMMUNICATION SKILLS": "Interpersonal and Communication Skills",
    "INTERPROFESSIONAL COLLABORATION": "Interprofessional Collaboration",
    "KNOWLEDGE FOR PRACTICE": "Knowledge for Practice",
    "SYSTEMS-BASED PRACTICE": "Systems-Based Practice",
    "PRACTICE-BASED LEARNING AND IMPROVEMENT": "Practice-Based Learning and Improvement",
}


def normalize_domain(value):
    s = " ".join((value or "").strip().split())
    return DOMAIN_ALIASES.get(s.upper(), s)


def parse_date(value):
    s = (value or "").strip()
    for fmt in ("%Y-%m-%d", "%m/%d/%Y", "%m/%d/%y"):
        try:
            return datetime.strptime(s, fmt).date().isoformat(), fmt
        except ValueError:
            pass
    return "", "unparsed"


def numeric_score_class(row):
    score_raw = (row["score"] or "").strip().upper()
    max_raw = (row["max_score"] or "").strip()
    if score_raw in {"P", "F"}:
        return "pass/fail score"
    if score_raw == "":
        return "missing score"
    try:
        score = float(score_raw)
    except ValueError:
        return "other non-numeric score"
    if max_raw == "":
        return "numeric score missing max_score"
    try:
        max_score = float(max_raw)
    except ValueError:
        return "non-numeric max_score"
    if max_score <= 0:
        return "non-positive max_score"
    if score < 0:
        return "negative score"
    if score > max_score:
        return "score above max_score"
    return "valid numeric in range"


for row in learners:
    row["learner_id_norm"] = normalize_learner_id(row["learner_id"])
    row["cohort_norm"] = normalize_cohort(row["cohort"])
for row in sessions:
    row["session_id_norm"] = normalize_session_id(row["session_id"])
for row in crosswalk:
    row["session_id_norm"] = normalize_session_id(row["session_id"])
    row["competency_domain_norm"] = normalize_domain(row["competency_domain"])
for row in events:
    row["learner_id_norm"] = normalize_learner_id(row["learner_id"])
    row["session_id_norm"] = normalize_session_id(row["session_id"])
    row["assessment_date_norm"], row["assessment_date_format"] = parse_date(row["assessment_date"])
    row["score_class"] = numeric_score_class(row)

raw_column_counts = {
    "learners": len(read_csv("learners.csv")[0]),
    "curricular_experiences": len(read_csv("curricular_experiences.csv")[0]),
    "competency_crosswalk": len(read_csv("competency_crosswalk.csv")[0]),
    "assessment_events": len(read_csv("assessment_events.csv")[0]),
}

print("Dataset inventory")
print("dataset, rows, raw columns")
for name, data in [
    ("learners", learners),
    ("curricular_experiences", sessions),
    ("competency_crosswalk", crosswalk),
    ("assessment_events", events),
]:
    print(f"{name}, {len(data)}, {raw_column_counts[name]}")

print("\nLearner identifiers and cohort normalization")
print("raw learner rows:", len(learners))
print("unique normalized learner IDs:", len({r["learner_id_norm"] for r in learners}))
print("duplicate normalized learner IDs:", {k: v for k, v in Counter(r["learner_id_norm"] for r in learners).items() if v > 1})
print("duplicate email groups:", {k: v for k, v in Counter(r["email"] for r in learners).items() if v > 1})
print("normalized cohort counts:", Counter(r["cohort_norm"] or "(missing)" for r in learners).most_common())
print("matriculation_year counts:", Counter(r["matriculation_year"] or "(missing)" for r in learners).most_common())
print("status counts:", Counter(r["status"] for r in learners).most_common())

print("\nSession and competency crosswalk")
print("unique normalized session IDs:", len({r["session_id_norm"] for r in sessions}))
print("duplicate normalized session IDs:", {k: v for k, v in Counter(r["session_id_norm"] for r in sessions).items() if v > 1})
print("sessions without crosswalk after normalization:", sorted({r["session_id_norm"] for r in sessions} - {r["session_id_norm"] for r in crosswalk}))
print("normalized competency domain counts:", Counter(r["competency_domain_norm"] for r in crosswalk).most_common())
print("multi-domain sessions:", sorted((k, v) for k, v in Counter(r["session_id_norm"] for r in crosswalk).items() if v > 1))

print("\nAssessment events")
print("unique event IDs:", len({r["event_id"] for r in events}))
print("assessment learner IDs missing from learners after normalization:", sorted({r["learner_id_norm"] for r in events} - {r["learner_id_norm"] for r in learners}))
print("assessment session IDs missing from sessions after normalization:", sorted({r["session_id_norm"] for r in events} - {r["session_id_norm"] for r in sessions}))
print("assessment date formats:", Counter(r["assessment_date_format"] for r in events).most_common())
print("parsed assessment date range:", min(r["assessment_date_norm"] for r in events if r["assessment_date_norm"]), "to", max(r["assessment_date_norm"] for r in events if r["assessment_date_norm"]))
print("dates outside expected 2024-08-01 to 2025-06-30 window:")
for row in events:
    if row["assessment_date_norm"] and (row["assessment_date_norm"] < "2024-08-01" or row["assessment_date_norm"] > "2025-06-30"):
        print(row["event_id"], row["learner_id"], row["session_id"], row["assessment_date"], "->", row["assessment_date_norm"])
print("score classification counts:", Counter(r["score_class"] for r in events).most_common())
print("score outlier examples:")
for row in events:
    if row["score_class"] in {"negative score", "score above max_score"}:
        print(row["event_id"], row["learner_id"], row["session_id"], row["score"], row["max_score"], row["score_class"])


Dataset inventory
dataset, rows, raw columns
learners, 154, 7
curricular_experiences, 22, 5
competency_crosswalk, 24, 2
assessment_events, 1751, 7

Learner identifiers and cohort normalization
raw learner rows: 154
unique normalized learner IDs: 150
duplicate normalized learner IDs: {'VU0024': 2, 'VU0142': 2, 'VU0105': 2, 'VU0004': 2}
duplicate email groups: {'hana.reed@vumc.example.edu': 2, 'dia.levi@vumc.example.edu': 2, 'ben.singh@vumc.example.edu': 2, 'kai.carter@vumc.example.edu': 2}
normalized cohort counts: [('2025', 39), ('2026', 38), ('2027', 38), ('2028', 35), ('(missing)', 4)]
matriculation_year counts: [('2024', 37), ('2022', 37), ('2021', 36), ('2023', 35), ('(missing)', 7), ('9999', 1), ('1900', 1)]
status counts: [('Active', 153), ('Withdrawn', 1)]

Session and competency crosswalk
unique normalized session IDs: 21
duplicate normalized session IDs: {'S005': 2}
sessions without crosswalk after normalization: ['S014', 'S018', 'S021']
normalized competency domain counts: [(

## Governance notes from the profile

The raw files are small, but they show realistic source-system messiness. The model should preserve raw values in staging tables and add normalized fields plus quality flags rather than silently changing source values.

Initial modeling decisions to carry forward:

- Normalize learner IDs for joins by trimming whitespace, uppercasing, removing hyphens, and zero-padding numeric IDs into `VU####`.
- Normalize session IDs by trimming and uppercasing, which resolves the `S021` / `s021` case mismatch.
- Normalize cohort to a four-digit year when present, but flag missing cohorts rather than imputing them.
- Map competency aliases such as `KP`, `PC`, and `IPC` to canonical domain labels with documented assumptions.
- Preserve pass/fail professionalism events separately from numeric scored assessments unless a defensible rubric is supplied.
- Treat `9999` matriculation year as a likely integer null sentinel and `1900` as suspicious, but do not alter them without flagging.
- Flag `999/25`, `105/100`, and `-3/20` as score quality cases. `105/100` may reflect extra credit; it should not be assumed invalid without a rule. For a conservative first-pass cohort benchmark, scores outside `[0, max_score]` are excluded rather than capped.
- Flag the 1999 assessment date as outside the expected curricular window.


In [2]:
learner_ids = {r["learner_id_norm"] for r in learners}
active_ids = {r["learner_id_norm"] for r in learners if r["status"] == "Active"}
cohort_by_learner = {r["learner_id_norm"]: r["cohort_norm"] for r in learners if r["cohort_norm"]}
domains_by_session = defaultdict(set)
for row in crosswalk:
    domains_by_session[row["session_id_norm"]].add(row["competency_domain_norm"])

valid_indicator_rows = []
exclusions = Counter()
for row in events:
    learner_id = row["learner_id_norm"]
    session_id = row["session_id_norm"]
    if learner_id not in learner_ids:
        exclusions["orphan learner"] += 1
        continue
    if learner_id not in active_ids:
        exclusions["non-active learner"] += 1
        continue
    if session_id not in domains_by_session:
        exclusions["no competency crosswalk"] += 1
        continue
    if row["score_class"] != "valid numeric in range":
        exclusions["not valid numeric in [0, max]"] += 1
        continue
    cohort = cohort_by_learner.get(learner_id)
    if not cohort:
        exclusions["missing cohort"] += 1
        continue
    score_pct = float(row["score"]) / float(row["max_score"])
    for domain in domains_by_session[session_id]:
        valid_indicator_rows.append((learner_id, cohort, domain, score_pct))

learner_domain_counts = Counter((learner_id, domain) for learner_id, cohort, domain, pct in valid_indicator_rows)
print("First-pass indicator feasibility, using conservative valid numeric scores only")
print("valid assessment-competency rows:", len(valid_indicator_rows))
print("learners with at least one valid scored competency row:", len({learner_id for learner_id, cohort, domain, pct in valid_indicator_rows}))
print("learner-domain pairs with at least two valid scores:", sum(1 for count in learner_domain_counts.values() if count >= 2))
print("learner-domain pairs with one valid score:", sum(1 for count in learner_domain_counts.values() if count == 1))
print("excluded assessment events before crosswalk expansion:", exclusions.most_common())


First-pass indicator feasibility, using conservative valid numeric scores only
valid assessment-competency rows: 1713
learners with at least one valid scored competency row: 145
learner-domain pairs with at least two valid scores: 349
learner-domain pairs with one valid score: 333
excluded assessment events before crosswalk expansion: [('no competency crosswalk', 306), ('not valid numeric in [0, max]', 45), ('missing cohort', 40), ('non-active learner', 12), ('orphan learner', 3)]


## First-pass indicator implication

A cohort-relative competency progression gap appears feasible, but it should be bounded by evidence coverage. The dashboard should distinguish:

- learner-domain pairs with enough scored evidence for cohort comparison;
- learner-domain pairs with limited evidence;
- assessment events excluded from the indicator because of missing/ambiguous scores, missing competency crosswalks, missing cohorts, or orphan learner IDs.

This supports a director-facing "strengths and monitor" view: show where learners are above/near benchmark, where learner-domain gaps are large enough to monitor, and where there is not enough evidence to interpret.
